In [10]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [11]:
!pip install wfdb neurokit2 numpy scipy scikit-learn xgboost torch tqdm


In [21]:
import os, random
import numpy as np
from tqdm import tqdm

import wfdb
import neurokit2 as nk
from scipy.signal import butter, filtfilt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb


In [22]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---- PTB-XL specific (SHORT ECGs) ----
N_WINDOW = 5
H_HORIZON = 2

RR_MIN, RR_MAX = 0.4, 1.8

BP_LOW, BP_HIGH = 0.5, 35.0
BP_ORDER = 2

BATCH_SIZE = 256
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-5

DATA_DIR = "/kaggle/input/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"


DEVICE: cuda


In [14]:
def bandpass_filter(sig, fs, low, high, order):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, sig)


In [15]:
def read_records(data_dir):
    with open(os.path.join(data_dir, "RECORDS"), "r") as f:
        return [line.strip() for line in f if line.strip()]


In [24]:
def pick_lead_ii(sig_names):
    for i, name in enumerate(sig_names):
        if name.strip().upper() == "II":
            return i
    return 1  # fallback


In [25]:
def extract_rr_from_ptbxl(base_dir, record):
    try:
        rec = wfdb.rdrecord(os.path.join(base_dir, record))
        fs = float(rec.fs)

        sig = rec.p_signal
        if sig is None or sig.shape[0] < fs * 6:
            return None

        lead_idx = pick_lead_ii(rec.sig_name)
        ecg = sig[:, lead_idx].astype(np.float32)

        # 🔴 PTB-XL amplitude normalization
        ecg = (ecg - np.mean(ecg)) / (np.std(ecg) + 1e-8)

        ecg_f = bandpass_filter(ecg, fs, BP_LOW, BP_HIGH, BP_ORDER)

        _, info = nk.ecg_peaks(ecg_f, sampling_rate=fs, method="neurokit")
        rpeaks = info.get("ECG_R_Peaks")

        if rpeaks is None or len(rpeaks) < 6:
            return None

        rr = np.diff(rpeaks) / fs
        rr = rr[(rr >= RR_MIN) & (rr <= RR_MAX)]

        if len(rr) < (N_WINDOW + H_HORIZON):
            return None

        return rr.astype(np.float32)

    except Exception:
        return None


In [26]:
def build_windows(rr, N, H):
    X, Y = [], []
    for i in range(len(rr) - N - H + 1):
        X.append(rr[i:i+N])
        Y.append(rr[i+N:i+N+H])
    return np.array(X), np.array(Y)


In [27]:
records = read_records(DATA_DIR)
print("Total PTB-XL records:", len(records))

X_all, Y_all, rec_ids = [], [], []

for r in tqdm(records):
    rr = extract_rr_from_ptbxl(DATA_DIR, r)
    if rr is None:
        continue

    Xr, Yr = build_windows(rr, N_WINDOW, H_HORIZON)
    if len(Xr) == 0:
        continue

    X_all.append(Xr)
    Y_all.append(Yr)
    rec_ids.extend([r] * len(Xr))

print("Valid records:", len(np.unique(rec_ids)))
print("Total windows:", len(rec_ids))

X_all = np.concatenate(X_all)
Y_all = np.concatenate(Y_all)
rec_ids = np.array(rec_ids)


Total PTB-XL records: 43597


100%|██████████| 43597/43597 [03:46<00:00, 192.15it/s]


Valid records: 43107
Total windows: 212101


In [28]:
unique_recs = np.unique(rec_ids)

train_r, test_r = train_test_split(unique_recs, test_size=0.15, random_state=SEED)
train_r, val_r  = train_test_split(train_r, test_size=0.15, random_state=SEED)

def mask(r):
    return np.isin(rec_ids, r)

X_train, Y_train = X_all[mask(train_r)], Y_all[mask(train_r)]
X_val,   Y_val   = X_all[mask(val_r)],   Y_all[mask(val_r)]
X_test,  Y_test  = X_all[mask(test_r)],  Y_all[mask(test_r)]


In [29]:
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1,1))

def scale(x):
    return scaler.transform(x.reshape(-1,1)).reshape(x.shape)

X_train_s = scale(X_train)
X_val_s   = scale(X_val)
X_test_s  = scale(X_test)

Y_train_s = scale(Y_train)
Y_val_s   = scale(Y_val)
Y_test_s  = scale(Y_test)


In [30]:
class RRDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]


In [31]:
train_loader = DataLoader(RRDataset(X_train_s, Y_train_s), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(RRDataset(X_val_s, Y_val_s), batch_size=BATCH_SIZE)
test_loader  = DataLoader(RRDataset(X_test_s, Y_test_s), batch_size=BATCH_SIZE)


In [32]:
class TransformerRegressor(nn.Module):
    def __init__(self, N, H, d_model=64, nhead=4):
        super().__init__()
        self.inp = nn.Linear(1, d_model)
        self.pos = nn.Parameter(torch.zeros(N, d_model))
        enc = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, 2)
        self.head = nn.Linear(d_model, H)

    def forward(self, x):
        x = x.unsqueeze(-1)
        z = self.inp(x) + self.pos.unsqueeze(0)
        z = self.encoder(z).mean(dim=1)
        return self.head(z)


In [33]:
def train_model(model):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    best, best_state = 1e9, None

    for ep in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                preds.append(model(xb.to(DEVICE)).cpu().numpy())
                trues.append(yb.numpy())

        rmse = np.sqrt(np.mean((np.concatenate(preds) - np.concatenate(trues))**2))
        print(f"Epoch {ep+1:02d} | Val RMSE: {rmse:.4f}")

        if rmse < best:
            best = rmse
            best_state = model.state_dict()

    model.load_state_dict(best_state)
    return model


In [34]:
model = TransformerRegressor(N_WINDOW, H_HORIZON)
model = train_model(model)

model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())

preds = np.concatenate(preds)

Epoch 01 | Val RMSE: 0.5491
Epoch 02 | Val RMSE: 0.5434
Epoch 03 | Val RMSE: 0.5486
Epoch 04 | Val RMSE: 0.5443
Epoch 05 | Val RMSE: 0.5352
Epoch 06 | Val RMSE: 0.5354
Epoch 07 | Val RMSE: 0.5355
Epoch 08 | Val RMSE: 0.5352
Epoch 09 | Val RMSE: 0.5357
Epoch 10 | Val RMSE: 0.5285
Epoch 11 | Val RMSE: 0.5358
Epoch 12 | Val RMSE: 0.5322
Epoch 13 | Val RMSE: 0.5310
Epoch 14 | Val RMSE: 0.5286
Epoch 15 | Val RMSE: 0.5365


In [35]:
def inv(x):
    return scaler.inverse_transform(x.reshape(-1,1)).reshape(x.shape)

y_true = inv(Y_test_s)
y_pred = inv(preds)

rmse_sec = np.sqrt(np.mean((y_true - y_pred)**2))
print("✅ Test RMSE (seconds):", rmse_sec)


✅ Test RMSE (seconds): 0.08887717


In [40]:
mae_sec = np.mean(np.abs(y_true - y_pred))
print("Test MAE (seconds):", mae_sec)


Test MAE (seconds): 0.047071703


In [41]:
mape = np.mean(np.abs((y_true - y_pred)/y_true)) * 100
print("Test MAPE (%):", mape)


Test MAPE (%): 6.5689087


In [42]:
for h in range(H_HORIZON):
    rmse_h = np.sqrt(np.mean((y_true[:,h] - y_pred[:,h])**2))
    print(f"Horizon {h+1} RMSE: {rmse_h:.4f} seconds")


Horizon 1 RMSE: 0.0845 seconds
Horizon 2 RMSE: 0.0931 seconds


In [49]:
# ==========================================================
# PTB-XL ECG RR Prediction Pipeline
# Transformer + GAT + XGBoost + Ensemble
# ==========================================================

import os, random
import numpy as np
from tqdm import tqdm
import wfdb
import neurokit2 as nk
from scipy.signal import butter, filtfilt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb

# ---------------- CONFIG ----------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

N_WINDOW = 5        # input RR window
H_HORIZON = 2       # predict next 2 RR values
RR_MIN, RR_MAX = 0.4, 1.8
BP_LOW, BP_HIGH = 0.5, 35.0
BP_ORDER = 2
BATCH_SIZE = 256
EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-5

DATA_DIR = "/kaggle/input/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

# ---------------- HELPERS ----------------
def bandpass_filter(sig, fs, low, high, order):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, sig)

def read_records(base_dir):
    # PTB-XL RECORDS file contains relative paths like records500/10001
    with open(os.path.join(base_dir,"RECORDS"),"r") as f:
        return [line.strip() for line in f if line.strip()]

def pick_lead_ii(sig_names):
    for i, name in enumerate(sig_names):
        if name.strip().upper() == "II":
            return i
    return 1

def extract_rr_from_ptbxl(base_dir, record):
    try:
        rec = wfdb.rdrecord(os.path.join(base_dir, record))
        fs = float(rec.fs)
        sig = rec.p_signal
        if sig is None or sig.shape[0] < fs*6:  # skip too short
            return None
        lead_idx = pick_lead_ii(rec.sig_name)
        ecg = sig[:, lead_idx].astype(np.float32)
        ecg = (ecg - np.mean(ecg)) / (np.std(ecg)+1e-8)
        ecg_f = bandpass_filter(ecg, fs, BP_LOW, BP_HIGH, BP_ORDER)
        _, info = nk.ecg_peaks(ecg_f, sampling_rate=fs, method="neurokit")
        rpeaks = info.get("ECG_R_Peaks")
        if rpeaks is None or len(rpeaks)<6:
            return None
        rr = np.diff(rpeaks)/fs
        rr = rr[(rr>=RR_MIN) & (rr<=RR_MAX)]
        if len(rr) < N_WINDOW + H_HORIZON:
            return None
        return rr.astype(np.float32)
    except:
        return None

def build_windows(rr, N, H):
    X, Y = [], []
    for i in range(len(rr) - N - H + 1):
        X.append(rr[i:i+N])
        Y.append(rr[i+N:i+N+H])
    return np.array(X), np.array(Y)

# ---------------- LOAD DATA ----------------
records = read_records(DATA_DIR)
print("Total PTB-XL records:", len(records))

X_all, Y_all, rec_ids = [], [], []

for r in tqdm(records):
    rr = extract_rr_from_ptbxl(DATA_DIR, r)
    if rr is None: continue
    Xr, Yr = build_windows(rr, N_WINDOW, H_HORIZON)
    if len(Xr)==0: continue
    X_all.append(Xr)
    Y_all.append(Yr)
    rec_ids.extend([r]*len(Xr))

if len(X_all)==0:
    raise ValueError("No valid RR sequences found! Check dataset path and RECORDS file.")

X_all = np.concatenate(X_all, axis=0)
Y_all = np.concatenate(Y_all, axis=0)
rec_ids = np.array(rec_ids)
print("Valid records:", len(np.unique(rec_ids)))
print("Total windows:", len(rec_ids))

unique_recs = np.unique(rec_ids)
train_r, test_r = train_test_split(unique_recs, test_size=0.15, random_state=SEED)
train_r, val_r  = train_test_split(train_r, test_size=0.15, random_state=SEED)

def mask(r): return np.isin(rec_ids, r)

X_train, Y_train = X_all[mask(train_r)], Y_all[mask(train_r)]
X_val,   Y_val   = X_all[mask(val_r)],   Y_all[mask(val_r)]
X_test,  Y_test  = X_all[mask(test_r)],  Y_all[mask(test_r)]

# Scale
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1,1))
def scale(x): return scaler.transform(x.reshape(-1,1)).reshape(x.shape)
X_train_s, X_val_s, X_test_s = scale(X_train), scale(X_val), scale(X_test)
Y_train_s, Y_val_s, Y_test_s = scale(Y_train), scale(Y_val), scale(Y_test)

# ---------------- DATASET ----------------
class RRDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

train_loader = DataLoader(RRDataset(X_train_s,Y_train_s), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(RRDataset(X_val_s,Y_val_s), batch_size=BATCH_SIZE)
test_loader  = DataLoader(RRDataset(X_test_s,Y_test_s), batch_size=BATCH_SIZE)

# ---------------- TRANSFORMER ----------------
class TransformerRegressor(nn.Module):
    def __init__(self, N,H,d_model=64,nhead=4):
        super().__init__()
        self.inp = nn.Linear(1,d_model)
        self.pos = nn.Parameter(torch.zeros(N,d_model))
        enc = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, 2)
        self.head = nn.Linear(d_model,H)
    def forward(self,x):
        x = x.unsqueeze(-1)
        z = self.inp(x)+self.pos.unsqueeze(0)
        z = self.encoder(z).mean(dim=1)
        return self.head(z)

# ---------------- GAT ----------------
class GraphAttentionLayer(nn.Module):
    def __init__(self,in_dim,out_dim,heads,dropout):
        super().__init__()
        self.out_dim = out_dim
        self.heads = heads
        self.W = nn.Linear(in_dim, out_dim*heads, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads,out_dim))
        self.a_dst = nn.Parameter(torch.empty(heads,out_dim))
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)
        self.leaky = nn.LeakyReLU(0.2)
        self.drop = nn.Dropout(dropout)
    def forward(self,x,adj):
        B,N,_ = x.shape
        h = self.W(x).view(B,N,self.heads,self.out_dim)
        s = (h*self.a_src.view(1,1,self.heads,self.out_dim)).sum(-1)
        d = (h*self.a_dst.view(1,1,self.heads,self.out_dim)).sum(-1)
        e = self.leaky(s.permute(0,2,1).unsqueeze(-1)+d.permute(0,2,1).unsqueeze(-2))
        mask = (adj==0).unsqueeze(0).unsqueeze(0)
        e = e.masked_fill(mask,float("-inf"))
        alpha = torch.softmax(e,dim=-1)
        alpha = self.drop(alpha)
        h_head = h.permute(0,2,1,3)
        out = torch.matmul(alpha,h_head)
        out = out.permute(0,2,1,3).contiguous().view(B,N,self.heads*self.out_dim)
        return out

class GATRegressor(nn.Module):
    def __init__(self,N,H,hidden=64,heads=4,layers=2,dropout=0.1):
        super().__init__()
        assert hidden%heads==0
        self.in_proj = nn.Linear(2,hidden)
        self.gats = nn.ModuleList()
        for _ in range(layers):
            self.gats.append(GraphAttentionLayer(hidden,hidden//heads,heads,dropout))
        self.act = nn.ELU()
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,H))
        adj = np.zeros((N,N),dtype=np.float32)
        for i in range(N):
            adj[i,i]=1
            if i-1>=0: adj[i,i-1]=1
            if i+1<N: adj[i,i+1]=1
        self.register_buffer("adj",torch.tensor(adj,dtype=torch.float32))
    def forward(self,x):
        rr = x
        drr = torch.zeros_like(rr)
        drr[:,1:] = rr[:,1:] - rr[:,:-1]
        feats = torch.stack([rr,drr],dim=-1)
        h = self.in_proj(feats)
        for layer in self.gats:
            h = self.act(layer(h,self.adj))
        h = h.mean(dim=1)
        h = self.drop(h)
        return self.head(h)

# ---------------- TRAIN ----------------
def train_model(model, train_loader=train_loader, val_loader=val_loader, epochs=EPOCHS):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    best, best_state = 1e9, None
    for ep in range(epochs):
        model.train()
        for xb,yb in train_loader:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        model.eval()
        preds,trues=[],[]
        with torch.no_grad():
            for xb,yb in val_loader:
                preds.append(model(xb.to(DEVICE)).cpu().numpy())
                trues.append(yb.numpy())
        rmse_val = np.sqrt(np.mean((np.concatenate(preds)-np.concatenate(trues))**2))
        print(f"Epoch {ep+1:02d} | Val RMSE: {rmse_val:.4f}")
        if rmse_val<best:
            best = rmse_val
            best_state = model.state_dict()
    model.load_state_dict(best_state)
    return model

def predict_model(model, loader=test_loader):
    model.eval()
    preds=[]
    with torch.no_grad():
        for xb,_ in loader:
            preds.append(model(xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)

def horizon_metrics(y_true,y_pred):
    res = []
    for h in range(H_HORIZON):
        res.append(np.sqrt(np.mean((y_true[:,h]-y_pred[:,h])**2)))
    return res

# ---------------- TRANSFORMER ----------------
tr = TransformerRegressor(N_WINDOW,H_HORIZON)
tr = train_model(tr)
pred_tr_test = predict_model(tr)
print("Transformer RMSE per horizon:", horizon_metrics(Y_test_s, pred_tr_test))

# ---------------- GAT ----------------
gat = GATRegressor(N_WINDOW,H_HORIZON)
gat = train_model(gat)
pred_gat_test = predict_model(gat)
print("GAT RMSE per horizon:", horizon_metrics(Y_test_s, pred_gat_test))

# ---------------- XGBOOST ----------------
def features_from_window(x):
    diffs = np.diff(x)
    rmssd = np.sqrt(np.mean(diffs**2)) if diffs.size else 0.0
    pnn50 = np.mean(np.abs(diffs)>0.05) if diffs.size else 0.0
    slope = np.polyfit(np.arange(len(x)),x,1)[0] if len(x)>=2 else 0.0
    return np.array([x[-1], np.mean(x), np.median(x), np.std(x), np.min(x), np.max(x),
                     x[-1]-x[-2] if len(x)>=2 else 0.0, rmssd, pnn50, slope],dtype=np.float32)

def make_xgb_matrix(X_seq):
    return np.stack([features_from_window(x) for x in X_seq],axis=0)

X_train_xgb = make_xgb_matrix(X_train_s)
X_val_xgb   = make_xgb_matrix(X_val_s)
X_test_xgb  = make_xgb_matrix(X_test_s)

base_xgb = xgb.XGBRegressor(n_estimators=800,max_depth=6,learning_rate=0.03,
                            subsample=0.9,colsample_bytree=0.9,reg_lambda=1.0,
                            objective="reg:squarederror",random_state=SEED,n_jobs=-1)

xgb_mo = MultiOutputRegressor(base_xgb)
xgb_mo.fit(X_train_xgb,Y_train_s)
pred_xgb_test = xgb_mo.predict(X_test_xgb).astype(np.float32)
print("XGBoost RMSE per horizon:", horizon_metrics(Y_test_s,pred_xgb_test))

# ---------------- ENSEMBLES ----------------
def median_ensemble(p1,p2,p3):
    return np.median(np.stack([p1,p2,p3],axis=0),axis=0).astype(np.float32)

def weighted_ensemble(preds,rmses):
    inv = np.array([1.0/max(r,1e-12) for r in rmses])
    w = inv/inv.sum()
    out = np.zeros_like(preds[0],dtype=np.float64)
    for wi,pi in zip(w,preds):
        out += wi*pi
    return out.astype(np.float32), w

pred_med_test = median_ensemble(pred_xgb_test,pred_gat_test,pred_tr_test)
pred_wavg_test, w = weighted_ensemble([pred_xgb_test,pred_gat_test,pred_tr_test],
                                     [np.sqrt(np.mean((Y_val_s-xgb_mo.predict(X_val_xgb))**2)),
                                      np.sqrt(np.mean((Y_val_s-predict_model(gat,val_loader))**2)),
                                      np.sqrt(np.mean((Y_val_s-predict_model(tr,val_loader))**2))])

print("Median Ensemble RMSE:", horizon_metrics(Y_test_s,pred_med_test))
print("Weighted Ensemble RMSE:", horizon_metrics(Y_test_s,pred_wavg_test))
print("Weighted Ensemble weights:", w)

# ---------------- CONVERT TO SECONDS ----------------
def inv_scale(arr): return scaler.inverse_transform(arr.reshape(-1,1)).reshape(arr.shape).astype(np.float32)

Y_test_sec = inv_scale(Y_test_s)
print("Transformer RMSE (s):", horizon_metrics(Y_test_sec, inv_scale(pred_tr_test)))
print("GAT RMSE (s):", horizon_metrics(Y_test_sec, inv_scale(pred_gat_test)))
print("XGB RMSE (s):", horizon_metrics(Y_test_sec, inv_scale(pred_xgb_test)))
print("Median Ensemble RMSE (s):", horizon_metrics(Y_test_sec, inv_scale(pred_med_test)))
print("Weighted Ensemble RMSE (s):", horizon_metrics(Y_test_sec, inv_scale(pred_wavg_test)))


DEVICE: cuda
Total PTB-XL records: 43597


100%|██████████| 43597/43597 [05:00<00:00, 144.90it/s]


Valid records: 43107
Total windows: 212101
Epoch 01 | Val RMSE: 0.5492
Epoch 02 | Val RMSE: 0.5436
Epoch 03 | Val RMSE: 0.5466
Epoch 04 | Val RMSE: 0.5443
Epoch 05 | Val RMSE: 0.5355
Epoch 06 | Val RMSE: 0.5353
Epoch 07 | Val RMSE: 0.5357
Epoch 08 | Val RMSE: 0.5395
Epoch 09 | Val RMSE: 0.5380
Epoch 10 | Val RMSE: 0.5312
Epoch 11 | Val RMSE: 0.5372
Epoch 12 | Val RMSE: 0.5319
Epoch 13 | Val RMSE: 0.5335
Epoch 14 | Val RMSE: 0.5294
Epoch 15 | Val RMSE: 0.5376
Transformer RMSE per horizon: [np.float32(0.5046643), np.float32(0.5507933)]
Epoch 01 | Val RMSE: 0.5435
Epoch 02 | Val RMSE: 0.5444
Epoch 03 | Val RMSE: 0.5425
Epoch 04 | Val RMSE: 0.5407
Epoch 05 | Val RMSE: 0.5408
Epoch 06 | Val RMSE: 0.5399
Epoch 07 | Val RMSE: 0.5392
Epoch 08 | Val RMSE: 0.5376
Epoch 09 | Val RMSE: 0.5386
Epoch 10 | Val RMSE: 0.5367
Epoch 11 | Val RMSE: 0.5383
Epoch 12 | Val RMSE: 0.5355
Epoch 13 | Val RMSE: 0.5364
Epoch 14 | Val RMSE: 0.5348
Epoch 15 | Val RMSE: 0.5342
GAT RMSE per horizon: [np.float32(0.5014

In [52]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

def features_from_window(x):
    diffs = np.diff(x)
    rmssd = np.sqrt(np.mean(diffs**2)) if diffs.size else 0.0
    pnn50 = np.mean(np.abs(diffs) > 0.05) if diffs.size else 0.0
    t = np.arange(len(x), dtype=np.float32)
    slope = np.polyfit(t, x, 1)[0] if len(x) >= 2 else 0.0
    return np.array([
        x[-1],
        np.mean(x),
        np.median(x),
        np.std(x),
        np.min(x),
        np.max(x),
        x[-1] - x[-2] if len(x) >= 2 else 0.0,
        rmssd,
        pnn50,
        slope
    ], dtype=np.float32)

def make_xgb_matrix(X_seq):
    return np.stack([features_from_window(x) for x in X_seq], axis=0).astype(np.float32)

X_train_xgb = make_xgb_matrix(X_train_s)
X_val_xgb   = make_xgb_matrix(X_val_s)
X_test_xgb  = make_xgb_matrix(X_test_s)

base_xgb = xgb.XGBRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

xgb_mo = MultiOutputRegressor(base_xgb)
xgb_mo.fit(X_train_xgb, Y_train_s)

pred_xgb_val  = xgb_mo.predict(X_val_xgb).astype(np.float32)
pred_xgb_test = xgb_mo.predict(X_test_xgb).astype(np.float32)

val_xgb_rmse_all = rmse(Y_val_s.reshape(-1), pred_xgb_val.reshape(-1))
print("✅ XGBoost done.")
print("XGB val RMSE_all:", val_xgb_rmse_all)
print("XGB test:", horizon_metrics(Y_test_s, pred_xgb_test))


✅ XGBoost done.
XGB val RMSE_all: 0.49957255
XGB test: [np.float32(0.46854308), np.float32(0.52838916)]


In [55]:
# ---------------- METRICS PER HORIZON ----------------
def horizon_results(y_true, y_pred):
    H = y_true.shape[1]
    rmse_k, mae_k = [], []
    for h in range(H):
        rmse_k.append(np.sqrt(np.mean((y_true[:,h]-y_pred[:,h])**2)))
        mae_k.append(np.mean(np.abs(y_true[:,h]-y_pred[:,h])))
    rmse_all = np.sqrt(np.mean((y_true - y_pred)**2))
    mae_all  = np.mean(np.abs(y_true - y_pred))
    return {
        'rmse_k': [float(x) for x in rmse_k],
        'mae_k': [float(x) for x in mae_k],
        'rmse_all': float(rmse_all),
        'mae_all': float(mae_all)
    }

# ---------------- CONVERT TO SECONDS ----------------
Y_test_sec = inv_scale(Y_test_s)
pred_tr_sec  = inv_scale(pred_tr_test)
pred_gat_sec = inv_scale(pred_gat_test)
pred_xgb_sec = inv_scale(pred_xgb_test)
pred_med_sec = inv_scale(pred_med_test)
pred_wavg_sec= inv_scale(pred_wavg_test)

# ---------------- RESULTS ----------------
results = {}

results['TR test']  = horizon_results(Y_test_sec, pred_tr_sec)
results['GAT test'] = horizon_results(Y_test_sec, pred_gat_sec)
results['XGB test'] = horizon_results(Y_test_sec, pred_xgb_sec)
results['MED test'] = horizon_results(Y_test_sec, pred_med_sec)
results['WAVG test']= horizon_results(Y_test_sec, pred_wavg_sec)

# ---------------- PRINT ----------------
for k,v in results.items():
    print(f"{k}: {v}")


TR test: {'rmse_k': [0.08502434194087982, 0.09279602020978928], 'mae_k': [0.04564618691802025, 0.04950324073433876], 'rmse_all': 0.08899504691362381, 'mae_all': 0.047574713826179504}
GAT test: {'rmse_k': [0.08447640389204025, 0.09223050624132156], 'mae_k': [0.04112963750958443, 0.0481264553964138], 'rmse_all': 0.088438481092453, 'mae_all': 0.044628042727708817}
XGB test: {'rmse_k': [0.07893874496221542, 0.08902143687009811], 'mae_k': [0.03644517809152603, 0.04520932957530022], 'rmse_all': 0.08413127064704895, 'mae_all': 0.04082725942134857}
MED test: {'rmse_k': [0.08132528513669968, 0.09059326350688934], 'mae_k': [0.03780499845743179, 0.04583447426557541], 'rmse_all': 0.08608409762382507, 'mae_all': 0.0418197400867939}
WAVG test: {'rmse_k': [0.08065278828144073, 0.09013387560844421], 'mae_k': [0.0381462424993515, 0.04559440538287163], 'rmse_all': 0.08552481234073639, 'mae_all': 0.041870322078466415}


In [56]:
def inv_scale(arr):
    return scaler.inverse_transform(arr.reshape(-1,1)).reshape(arr.shape).astype(np.float32)

Y_test_sec = inv_scale(Y_test_s)

xgb_sec = inv_scale(pred_xgb_test)
gat_sec = inv_scale(pred_gat_test)
tr_sec  = inv_scale(pred_tr_test)
med_sec = inv_scale(pred_med_test)
wavg_sec = inv_scale(pred_wavg_test)

print("XGB (sec):", horizon_metrics(Y_test_sec, xgb_sec))
print("GAT (sec):", horizon_metrics(Y_test_sec, gat_sec))
print("TR  (sec):", horizon_metrics(Y_test_sec, tr_sec))
print("MED (sec):", horizon_metrics(Y_test_sec, med_sec))
print("WAV (sec):", horizon_metrics(Y_test_sec, wavg_sec))


XGB (sec): [np.float32(0.078938745), np.float32(0.08902144)]
GAT (sec): [np.float32(0.084476404), np.float32(0.092230506)]
TR  (sec): [np.float32(0.08502434), np.float32(0.09279602)]
MED (sec): [np.float32(0.081325285), np.float32(0.09059326)]
WAV (sec): [np.float32(0.08065279), np.float32(0.090133876)]


In [58]:
# ---------------- ENSEMBLE FUNCTIONS ----------------
def median_ensemble(p1, p2, p3):
    return np.median(np.stack([p1,p2,p3], axis=0), axis=0).astype(np.float32)

def weighted_ensemble(preds, rmses):
    # Inverse RMSE weighting
    inv = np.array([1.0/max(r,1e-12) for r in rmses], dtype=np.float64)
    w = inv / inv.sum()
    out = np.zeros_like(preds[0], dtype=np.float64)
    for wi, pi in zip(w, preds):
        out += wi * pi
    return out.astype(np.float32), w.astype(np.float32)

# ---------------- VALIDATION RMSE PER MODEL ----------------
def model_val_rmse(model, X_val_loader, model_type="torch"):
    if model_type=="torch":
        preds = []
        model.eval()
        with torch.no_grad():
            for xb, yb in X_val_loader:
                preds.append(model(xb.to(DEVICE)).cpu().numpy())
        preds = np.concatenate(preds)
    elif model_type=="xgb":
        preds = xgb_mo.predict(X_val_xgb).astype(np.float32)
    else:
        raise ValueError("Invalid model_type")
    return np.sqrt(np.mean((Y_val_s - preds)**2))

val_tr_rmse_all  = model_val_rmse(tr, val_loader, "torch")
val_gat_rmse_all = model_val_rmse(gat, val_loader, "torch")
val_xgb_rmse_all = model_val_rmse(None, None, "xgb")  # XGB uses X_val_xgb internally

# ---------------- MEDIAN ENSEMBLE ----------------
pred_med_test = median_ensemble(pred_xgb_test, pred_gat_test, pred_tr_test)
print("MED test:", horizon_metrics(Y_test_s, pred_med_test))

# ---------------- WEIGHTED ENSEMBLE ----------------
pred_wavg_test, w = weighted_ensemble(
    [pred_xgb_test, pred_gat_test, pred_tr_test],
    [val_xgb_rmse_all, val_gat_rmse_all, val_tr_rmse_all]
)
print("WAVG weights (XGB, GAT, TR):", w)
print("WAVG test:", horizon_metrics(Y_test_s, pred_wavg_test))


MED test: [np.float32(0.48270848), np.float32(0.53771883)]
WAVG weights (XGB, GAT, TR): [0.34910208 0.32647172 0.3244262 ]
WAVG test: [np.float32(0.47871682), np.float32(0.5349921)]


**Persistance Baseline**

In [59]:
# ---------------- Persistence Baseline ----------------
# For each test sample, predict next H RR values as the last RR in the window

# X_test_s shape: (num_samples, N)
last_rr = X_test_s[:, -1]           # (num_samples,)
pred_persist_test = np.repeat(
    last_rr[:, None], H_HORIZON, axis=1
).astype(np.float32)

print("Persistence baseline (scaled):",
      horizon_metrics(Y_test_s, pred_persist_test))

# Convert to seconds
pred_persist_sec = scaler.inverse_transform(
    pred_persist_test.reshape(-1,1)
).reshape(pred_persist_test.shape)

Y_test_sec = scaler.inverse_transform(
    Y_test_s.reshape(-1,1)
).reshape(Y_test_s.shape)

print("Persistence baseline (seconds):",
      horizon_metrics(Y_test_sec, pred_persist_sec))


Persistence baseline (scaled): [np.float32(0.67714995), np.float32(0.69976234)]
Persistence baseline (seconds): [np.float32(0.11408421), np.float32(0.11789388)]


**LSTM**

In [60]:
import torch
import torch.nn as nn

class LSTMRegressor(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=1, horizon=5, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, horizon)
        )

    def forward(self, x):
        # x: (B, N)
        x = x.unsqueeze(-1)          # (B, N, 1)
        out, (h, c) = self.lstm(x)   # h: (num_layers, B, hidden_dim)
        h_last = h[-1]               # (B, hidden_dim)
        return self.head(h_last)     # (B, H)


In [61]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, wd=1e-5):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()

    best = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())

        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                ps.append(model(xb).cpu().numpy())
                ys.append(yb.numpy())
        y_true = np.concatenate(ys, 0)
        y_pred = np.concatenate(ps, 0)
        val_rmse_all = rmse(y_true.reshape(-1), y_pred.reshape(-1))

        if val_rmse_all < best:
            best = val_rmse_all
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {ep:02d} | train_loss={np.mean(losses):.6f} | val_RMSE_all={val_rmse_all:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best

def predict_model(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds, 0)


In [62]:
from torch.utils.data import Dataset, DataLoader

class RRSeqDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

train_ds = RRSeqDataset(X_train_s, Y_train_s)
val_ds   = RRSeqDataset(X_val_s, Y_val_s)
test_ds  = RRSeqDataset(X_test_s, Y_test_s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


In [63]:
lstm = LSTMRegressor(input_dim=1, hidden_dim=64, num_layers=1, horizon=H_HORIZON, dropout=0.0)
lstm, val_lstm_rmse_all = train_model(lstm, train_loader, val_loader, epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY)

pred_lstm_test = predict_model(lstm, test_loader)

print("LSTM val RMSE_all (scaled):", val_lstm_rmse_all)
print("LSTM test (scaled):", horizon_metrics(Y_test_s, pred_lstm_test))


Epoch 01 | train_loss=0.325037 | val_RMSE_all=0.546539
Epoch 02 | train_loss=0.290746 | val_RMSE_all=0.545456
Epoch 03 | train_loss=0.287112 | val_RMSE_all=0.539860
Epoch 04 | train_loss=0.282409 | val_RMSE_all=0.536009
Epoch 05 | train_loss=0.278645 | val_RMSE_all=0.532358
Epoch 06 | train_loss=0.275442 | val_RMSE_all=0.529467
Epoch 07 | train_loss=0.272886 | val_RMSE_all=0.527348
Epoch 08 | train_loss=0.271263 | val_RMSE_all=0.527165
Epoch 09 | train_loss=0.269950 | val_RMSE_all=0.526604
Epoch 10 | train_loss=0.268890 | val_RMSE_all=0.526159
Epoch 11 | train_loss=0.267612 | val_RMSE_all=0.524898
Epoch 12 | train_loss=0.266904 | val_RMSE_all=0.524111
Epoch 13 | train_loss=0.266179 | val_RMSE_all=0.523676
Epoch 14 | train_loss=0.265094 | val_RMSE_all=0.523073
Epoch 15 | train_loss=0.264536 | val_RMSE_all=0.523266
LSTM val RMSE_all (scaled): 0.5230732
LSTM test (scaled): [np.float32(0.48455065), np.float32(0.543624)]


In [64]:
def inv_scale(arr):
    return scaler.inverse_transform(arr.reshape(-1,1)).reshape(arr.shape).astype(np.float32)

Y_test_sec = inv_scale(Y_test_s)
lstm_sec = inv_scale(pred_lstm_test)

print("LSTM test (seconds):", horizon_metrics(Y_test_sec, lstm_sec))


LSTM test (seconds): [np.float32(0.081635654), np.float32(0.09158815)]


**Statistical significance 5 runs mean ± std paired test (p-value)**

In [66]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [67]:
from scipy.stats import ttest_rel

SEEDS = [0, 1, 2, 3, 4]

rmse_lstm_runs = []
rmse_ens_runs = []

for seed in SEEDS:
    print(f"\n===== RUN seed={seed} =====")
    set_seed(seed)

    # ---- LSTM ----
    lstm = LSTMRegressor(
        input_dim=1,
        hidden_dim=64,
        num_layers=1,
        horizon=H_HORIZON
    )
    lstm, _ = train_model(lstm, train_loader, val_loader, epochs=EPOCHS)
    pred_lstm = predict_model(lstm, test_loader)

    lstm_sec = scaler.inverse_transform(
        pred_lstm.reshape(-1,1)
    ).reshape(pred_lstm.shape)

    Y_test_sec = scaler.inverse_transform(
        Y_test_s.reshape(-1,1)
    ).reshape(Y_test_s.shape)

    rmse_lstm = rmse(Y_test_sec.reshape(-1), lstm_sec.reshape(-1))
    rmse_lstm_runs.append(rmse_lstm)

    # ---- ENSEMBLE (use your already-trained components or retrain if needed) ----
    # If retraining everything is too slow, retrain only Transformer+GAT once and reuse XGB

    pred_ens = inv_scale(pred_wavg_test)  # OR recompute per run if retraining
    rmse_ens = rmse(Y_test_sec.reshape(-1), pred_ens.reshape(-1))
    rmse_ens_runs.append(rmse_ens)

    print(f"LSTM RMSE: {rmse_lstm:.5f}, Ensemble RMSE: {rmse_ens:.5f}")



===== RUN seed=0 =====
Epoch 01 | train_loss=0.325572 | val_RMSE_all=0.546169
Epoch 02 | train_loss=0.290327 | val_RMSE_all=0.545326
Epoch 03 | train_loss=0.284671 | val_RMSE_all=0.537152
Epoch 04 | train_loss=0.279032 | val_RMSE_all=0.532070
Epoch 05 | train_loss=0.275661 | val_RMSE_all=0.529366
Epoch 06 | train_loss=0.272816 | val_RMSE_all=0.528909
Epoch 07 | train_loss=0.271445 | val_RMSE_all=0.526540
Epoch 08 | train_loss=0.270326 | val_RMSE_all=0.527486
Epoch 09 | train_loss=0.269300 | val_RMSE_all=0.525725
Epoch 10 | train_loss=0.268607 | val_RMSE_all=0.525874
Epoch 11 | train_loss=0.267341 | val_RMSE_all=0.524744
Epoch 12 | train_loss=0.266616 | val_RMSE_all=0.524375
Epoch 13 | train_loss=0.265762 | val_RMSE_all=0.524235
Epoch 14 | train_loss=0.265198 | val_RMSE_all=0.523111
Epoch 15 | train_loss=0.264568 | val_RMSE_all=0.522665
LSTM RMSE: 0.08661, Ensemble RMSE: 0.08552

===== RUN seed=1 =====
Epoch 01 | train_loss=0.327899 | val_RMSE_all=0.546567
Epoch 02 | train_loss=0.29012

In [68]:
import numpy as np

lstm_mean = np.mean(rmse_lstm_runs)
lstm_std  = np.std(rmse_lstm_runs)

ens_mean = np.mean(rmse_ens_runs)
ens_std  = np.std(rmse_ens_runs)

print(f"LSTM:     {lstm_mean:.4f} ± {lstm_std:.4f}")
print(f"Ensemble: {ens_mean:.4f} ± {ens_std:.4f}")


LSTM:     0.0867 ± 0.0001
Ensemble: 0.0855 ± 0.0000


In [69]:
t_stat, p_value = ttest_rel(rmse_lstm_runs, rmse_ens_runs)

print("Paired t-test p-value:", p_value)

Paired t-test p-value: 2.5175069158545896e-06


In [70]:
from scipy.stats import ttest_rel

# Models predictions (already computed)
preds_dict = {
    "TR": pred_tr_test,
    "GAT": pred_gat_test,
    "XGB": pred_xgb_test,
    "MED": pred_med_test,
    "WAVG": pred_wavg_test
}

# Compute RMSE and MAE per horizon
def compute_metrics(y_true, y_pred):
    rmse_h = np.sqrt(np.mean((y_true - y_pred)**2, axis=0))
    mae_h  = np.mean(np.abs(y_true - y_pred), axis=0)
    rmse_all = np.sqrt(np.mean((y_true - y_pred)**2))
    mae_all  = np.mean(np.abs(y_true - y_pred))
    return {"rmse_h": rmse_h, "mae_h": mae_h, "rmse_all": rmse_all, "mae_all": mae_all}

metrics_dict = {name: compute_metrics(Y_test_sec, inv_scale(preds)) for name, preds in preds_dict.items()}

# Paired t-test function
def paired_t_test(y_true, y_pred1, y_pred2):
    # flatten all horizons
    return ttest_rel(inv_scale(y_pred1).flatten(), inv_scale(y_pred2).flatten()).pvalue

# Comparison table
print(f"{'Model':<8} {'RMSE_all':<10} {'MAE_all':<10} {'p vs MED':<12} {'p vs WAVG':<12}")
for name, metrics in metrics_dict.items():
    if name in ["MED", "WAVG"]:
        p_med, p_wavg = "-", "-"
    else:
        p_med = paired_t_test(Y_test_sec, preds_dict[name], preds_dict["MED"])
        p_wavg = paired_t_test(Y_test_sec, preds_dict[name], preds_dict["WAVG"])
    print(f"{name:<8} {metrics['rmse_all']:<10.4f} {metrics['mae_all']:<10.4f} {p_med:<12} {p_wavg:<12}")


Model    RMSE_all   MAE_all    p vs MED     p vs WAVG   
TR       0.0890     0.0476     0.0          0.0         
GAT      0.0884     0.0446     0.01480896997806561 1.0641087753288636e-81
XGB      0.0841     0.0408     4.8515093671064125e-113 0.0         
MED      0.0861     0.0418     -            -           
WAVG     0.0855     0.0419     -            -           
